# Donut fine-tune - InBody **270 + 570** v7 - Kaggle runner

Retrains Donut on the **v7** both-device synthetic set (2500 `inbody_270` + 2500 `inbody_570`).

**What v7 changes, and what it is testing.** A real 270 prints arm and leg lean mass to two
decimals. Every earlier synthetic sheet printed one, so no arm target was ever longer than three
characters, and on real photos v5 and v6 sometimes return an arm with its last digit cut off
(#59). v7's 270 prints and labels its four limbs to two decimals, `3.40` included (ADR-0007,
2026-09-15 amendment). Trunk stays at one decimal, as on a real sheet.

**Nothing else changed.** The 570 half renders exactly as v6's did, no geometry or augmentation
change, same recipe as v6: fresh from `donut-base`, 3 epochs.

### The prediction, written before the run

From #59, label shape against read shape on the real hold-out:

```
                     v5-3750            v6-2500
arm  #.## -> #.##    17 right           13 right
arm  #.## -> #.#      4 wrong, 1 right   4 wrong, 2 right
arm  #.## -> unread   0                  3
```

- If the arm-length habit is the cause, v7 reads no two-decimal arm as `#.#`, and the wrong
  one-decimal arm reads go to 0 while `#.## -> #.##` rises toward 22.
- If v7 still cuts arms short, the length-prior hypothesis in #59 is wrong and the cause is
  somewhere else (the print, the crop, the resolution), not the training target.
- Segmental lean and critical fields should not fall below v5-3750's.

Score it with ADR-0008's arm check still on, and also count label shape against read shape per
limb, as in the table above. The check turns a cut arm into a flag, so the outcome split alone
hides exactly the change being measured. Retire the check only if the shapes say so (ADR-0008).

**Budget the 12 h session cap.** v5 and v6 each finished 3 epochs on 5000 sheets well inside it,
and `processor.save_pretrained` only runs after `trainer.train()` returns, so a processor at the
run root is the cheapest proof the run finished.


In [ ]:
# GPU + the two flags from prior runs: pin to one GPU (dual-T4 OOMs donut-base on
# GPU0) and enable expandable segments to avoid fragmentation OOMs.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the branch carrying the two-decimal 270 limbs (ADR-0007, 2026-09-15).
BRANCH = 'feat/module-1-real-holdout-scorer'
from kaggle_secrets import UserSecretsClient
try:
    GH_TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
    REPO = f'https://{GH_TOKEN}@github.com/QeekOw/InForm.git'
except Exception:
    REPO = 'https://github.com/QeekOw/InForm.git'  # public fallback
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --branch $BRANCH --single-branch $REPO repo
%cd /kaggle/working/repo
!pip install -q -e '.[training]' gdown

In [ ]:
# Dataset attached as a Kaggle Dataset (Add Input, right panel) - no Drive/gdown.
# Find the sheets wherever Kaggle mounted them; if the input is still a .zip,
# extract it to /kaggle/tmp. Sets DATA_DIR to the folder holding the sheets.
import glob, os, shutil

# generate_dataset writes .jpg (inform.training.dataset.IMAGE_SUFFIXES); runs
# before that wrote .png, so accept either rather than silently finding nothing.
SUFFIXES = ('jpg', 'jpeg', 'png')

def find_sheets(root):
    return sorted(p for s in SUFFIXES for p in glob.glob(f'{root}/**/*.{s}', recursive=True))

sheets = find_sheets('/kaggle/input')
if not sheets:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No sheets or zip under /kaggle/input - click Add Input and attach your dataset.'
    shutil.unpack_archive(zips[0], '/kaggle/tmp')
    sheets = find_sheets('/kaggle/tmp')
assert sheets, 'No sheets found after extract - check the dataset contents.'
DATA_DIR = os.path.dirname(sheets[0])
n270 = len([p for p in sheets if 'inbody_270' in p]); n570 = len([p for p in sheets if 'inbody_570' in p])
print('DATA_DIR =', DATA_DIR, '| sheets:', len(sheets), '| 270:', n270, '570:', n570)


In [ ]:
# Fresh from donut-base, 3 epochs, BOTH devices. Do NOT resume from v6 or earlier:
# v7 changes the 270's limb targets from one decimal to two.
# batch 1 + grad-accum 4 fits the 2560x1920 canvas on a T4; effective batch 4.
# --batch-size 2 OOMs even on 16 GB - raise grad-accum instead.
# 4 dataloader workers keep the GPU fed while JPEGs decode.
# 3 epochs fits the 12 h cap, so the run completes and saves its processor.
CHECKPOINT_DIR = '/kaggle/working/donut-both-v7'
!python -m inform.training.train \
  --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 \
  --dataloader-num-workers 4 --learning-rate 3e-5


In [ ]:
# RESUME ONLY - run this instead of cell 4 when a previous session hit the 12 h cap.
# Attach that session's output as an input, copy the checkpoints into CHECKPOINT_DIR,
# then --resume picks up from the last checkpoint-* subdir.
#
# import glob, shutil, os
# prev = glob.glob('/kaggle/input/**/donut-both-v7', recursive=True)[0]
# shutil.copytree(prev, CHECKPOINT_DIR, dirs_exist_ok=True)
# !python -m inform.training.train \
#   --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR \
#   --model-name-or-path naver-clova-ix/donut-base \
#   --epochs 3 --batch-size 1 --gradient-accumulation-steps 4 \
#   --dataloader-num-workers 4 --learning-rate 3e-5 --resume


In [ ]:
# Every epoch checkpoint is kept (save_strategy='epoch', save_total_limit=None), so
# /kaggle/working holds checkpoint-* subdirs plus the final top-level model. All of it
# lands in the Save Version output. ~2.4 GB per checkpoint - watch the 20 GB /kaggle/working
# limit, and download to a drive with room (NOT C:, which has ~12 GB free).
!du -sh /kaggle/working/donut-both-v7/* | sort -h
!ls -la /kaggle/working/donut-both-v7


## After the run

Score **every** epoch checkpoint, not just the last, against v5-3750 (the default engine) and
v6. v6's epoch 2 beat its epoch 3, so the last epoch is not assumed best.

```bash
python -m inform.holdout --data-dir data/real_holdout \
    --labels data/real_holdout/labels.json \
    --donut-checkpoint /path/to/donut-both-v5/checkpoint-3750 \
    --donut-checkpoint /path/to/donut-both-v6/checkpoint-2500 \
    --donut-checkpoint /path/to/donut-both-v7/checkpoint-<N>
```

Per-epoch checkpoint dirs carry no processor; copy `processor_config.json`, `tokenizer.json` and
`tokenizer_config.json` from the run root into each one before loading it.

The baselines are in `docs/training.md` (n=12 table) and #59's comment (split and silent errors
before and after the arm check). Figures from before the arm check (94ad5a2) are not comparable
with later ones.

### Read the headline beside the outcome split, always

A silent error is a wrong value inside an `unverified` read. An engine that flags or under-reads
everything has no `unverified` reads and therefore a perfect headline, which is why the scorer
prints both together (CONTEXT.md).
